In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score

2026-02-07 20:53:45.391330: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-07 20:53:45.397006: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-07 20:53:45.945494: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-07 20:53:49.985261: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
features_map = {
    "J. Kampe": [
        "z_term_3", "z_term_2", "d_lag_2", "d_lag_3", "d_lag_4", "z_cogram", "d_lag_16", "d_lag_5",
        "d_lag_15", "d_lag_17", "d_lag_12", "d_lag_6", "d_lag_1", "z_term_4", "z_term_6", "d_lag_18",
        "d_lag_13", "d_lag_7", "d_lag_14", "z_cogram_lag_4", "z_gram_lag_4", "z_cogram_lag_5", "z_gram_lag_5", "z_gram_lag_7",
        "z_term_5", "z_gram_lag_6", "z_cogram_lag_6", "z_cogram_lag_7", "d_lag_11", "z_cogram_lag_2", "d_lag_21", "z_cogram_lag_8",
        "z_cogram_lag_11", "z_cogram_lag_3", "d_lag_8", "z_gram_lag_2", "z_gram_lag_8", "z_gram_lag_1", "z_cogram_lag_12", "d_lag_19"
    ]
}

random_forest_features = pd.read_csv("../results/random_forest_feature_selection.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/gevrey_method_feature_selection.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/mrmr_10_features.csv")["feature"].tolist()
mrmr_14_features = pd.read_csv("../results/mrmr_14_features.csv")["feature"].tolist()

features_map["Random Forest (5)"] = random_forest_features[:5]
features_map["Random Forest (10)"] = random_forest_features[:10]
features_map["Random Forest Full"] = random_forest_features
features_map["Correlation"] = correlation_features
features_map["Gevrey Method"] = gevrey_method_features
features_map["Gevrey Method (8 features)"] = gevrey_method_features[:8]    # Limiting to top 8 features
features_map["Gevrey Method (10 features)"] = gevrey_method_features[:10]  # Limiting to top 10 features
features_map["Gevrey Method (12 features)"] = gevrey_method_features[:12]  # Limiting to top 12 features
features_map["Gevrey Method (14 features)"] = gevrey_method_features[:14]  # Limiting to top 14 features
features_map["Gevrey Method (20 features)"] = gevrey_method_features[:20]  # Limiting to top 20 features
features_map["mRMR (10 features)"] = mrmr_10_features
features_map["mRMR (14 features)"] = mrmr_14_features

In [3]:
class DatasetScalerService:
    MAX_LIMIT = 100_000
    def __init__(self, features: list[str]):
        self.__scaler_X = MinMaxScaler()
        self.__scaler_y = MinMaxScaler()
        self.__X_original = pd.read_csv("../dataset/j_kampe.csv")
        self.__y_original = pd.read_csv("../dataset/distances.csv")["distance"]
        self.__features = features

    def get_scaled_data(self, limit: int = 11_000):
        if limit > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.__X_original[self.__features].values
        y = self.__y_original.values.reshape(-1, 1)

        X = X[1_000:limit]
        y = y[1_000:limit]

        X_train = X[:int(0.8 * X.shape[0])]
        X_test  = X[int(0.8 * X.shape[0]):]
        y_train = y[:int(0.8 * y.shape[0])]
        y_test  = y[int(0.8 * y.shape[0]):]

        X_train_scaled = self.__scaler_X.fit_transform(X_train)
        X_test_scaled  = self.__scaler_X.transform(X_test)
        y_train_scaled = self.__scaler_y.fit_transform(y_train).ravel()
        y_test_scaled  = self.__scaler_y.transform(y_test).ravel()
        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled

    @property
    def scaler_y(self) -> MinMaxScaler:
        return self.__scaler_y

In [4]:
C_values = (0.1, 1.0, 10.0)
epsilon_values = (0.01, 0.1, 1, 0.02)
best_results = []

for name, features in features_map.items():
    best_rmse = np.inf
    best_r2   = -np.inf
    best_result = None
    dataset_scaler_service = DatasetScalerService(features)
    (
        X_train_scaled,
        X_test_scaled,
        y_train_scaled,
        y_test_scaled
    ) = dataset_scaler_service.get_scaled_data()

    scaler_y = dataset_scaler_service.scaler_y

    for C in C_values:
        for epsilon in epsilon_values:
            model = SVR(kernel="rbf", C=C, epsilon=epsilon)
            model.fit(X_train_scaled, y_train_scaled)
            y_pred_scaled = model.predict(X_test_scaled)

            y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))
            y_test = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1))

            rmse = root_mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            if rmse < best_rmse and r2 > best_r2:
                best_rmse = rmse
                best_r2 = r2
                best_result = {
                    "Group": name,
                    "model": "SVR",
                    "Best RMSE": rmse,
                    "R2": r2,
                    "Best C": C,
                    "Best Epsilon": epsilon,
                    "Features": len(features)
                }
    if best_result:
        best_results.append(best_result)

svr_df = pd.DataFrame(best_results).sort_values("Best RMSE").reset_index(drop=True)
print(svr_df)
svr_df.to_csv("../results/svr_experiment_5.csv", index=False)

                          Group model  Best RMSE        R2  Best C  \
0            Random Forest (10)   SVR   0.054611  0.954601    10.0   
1                      J. Kampe   SVR   0.061501  0.942423    10.0   
2             Random Forest (5)   SVR   0.067228  0.931201    10.0   
3   Gevrey Method (12 features)   SVR   0.070385  0.924587    10.0   
4                 Gevrey Method   SVR   0.081848  0.898024    10.0   
5   Gevrey Method (14 features)   SVR   0.090470  0.875407     1.0   
6                   Correlation   SVR   0.091206  0.873371    10.0   
7            Random Forest Full   SVR   0.095188  0.862076     1.0   
8   Gevrey Method (20 features)   SVR   0.099592  0.849016    10.0   
9            mRMR (10 features)   SVR   0.101834  0.842143    10.0   
10  Gevrey Method (10 features)   SVR   0.108313  0.821416    10.0   
11           mRMR (14 features)   SVR   0.118905  0.784779     1.0   
12   Gevrey Method (8 features)   SVR   0.122821  0.770372    10.0   

    Best Epsilon  F

In [5]:
best_results = []

for name, features in features_map.items():
    best_rmse = np.inf
    best_r2   = -np.inf
    best_result = None

    dataset_scaler_service = DatasetScalerService(features)

    X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled = dataset_scaler_service.get_scaled_data()
    scaler_y = dataset_scaler_service.scaler_y

    hidden_layer_sizes = [
        (34, 34),
        (100, 50),
        (100, 100, 50),
        (94,)
    ]

    for hls in hidden_layer_sizes:
        model = MLPRegressor(hidden_layer_sizes=hls, max_iter=500, random_state=42)
        model.fit(X_train_scaled, y_train_scaled)
        y_pred_scaled = model.predict(X_test_scaled)

        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))
        y_test = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1))

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        if rmse < best_rmse and r2 > best_r2:
            best_rmse = rmse
            best_r2 = r2
            best_result = {
                "Group": name,
                "model": "MLP",
                "Best RMSE": rmse,
                "R2": r2,
                "Best Hidden Layer Sizes": hls,
                "Features": len(features)
            }

    if best_result:
        best_results.append(best_result)

mlp_df = pd.DataFrame(best_results).sort_values("Best RMSE").reset_index(drop=True)
print(mlp_df)
mlp_df.to_csv("../results/mlp_experiment_5.csv", index=False)

                          Group model  Best RMSE        R2  \
0                      J. Kampe   MLP   0.060677  0.943955   
1            mRMR (14 features)   MLP   0.062800  0.939966   
2                 Gevrey Method   MLP   0.065202  0.935285   
3             Random Forest (5)   MLP   0.067710  0.930211   
4            Random Forest Full   MLP   0.072926  0.919044   
5   Gevrey Method (20 features)   MLP   0.074180  0.916235   
6            mRMR (10 features)   MLP   0.074771  0.914896   
7   Gevrey Method (14 features)   MLP   0.074913  0.914572   
8            Random Forest (10)   MLP   0.079467  0.903871   
9   Gevrey Method (12 features)   MLP   0.084197  0.892087   
10  Gevrey Method (10 features)   MLP   0.106736  0.826580   
11   Gevrey Method (8 features)   MLP   0.111450  0.810923   
12                  Correlation   MLP   0.150071  0.657174   

   Best Hidden Layer Sizes  Features  
0           (100, 100, 50)        40  
1           (100, 100, 50)        14  
2           (1

In [6]:
number_of_output_neuros = 1
number_of_hidden_neuros = 55
epochs = 100

results = []

for name, features in features_map.items():
    dataset_scaler_service = DatasetScalerService(features)
    X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled = dataset_scaler_service.get_scaled_data()

    input_shape = (X_train_scaled.shape[1], 1)

    rnn_model = Sequential()
    rnn_model.add(Dense(number_of_hidden_neuros, activation='relu', input_shape=(X_train_scaled.shape[1],)))
    rnn_model.add(Dense(number_of_hidden_neuros, activation='relu'))
    rnn_model.add(Dense(number_of_hidden_neuros, activation='relu'))
    rnn_model.add(Dense(number_of_hidden_neuros, activation='relu'))
    rnn_model.add(Dense(number_of_output_neuros, activation='sigmoid'))

    rnn_model.compile(optimizer='adam', loss='mean_squared_error')
    rnn_model.fit(X_train_scaled, y_train_scaled, epochs=epochs, batch_size=32, verbose=0)

    y_pred_scaled = rnn_model.predict(X_test_scaled).ravel()
    y_pred = dataset_scaler_service.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))
    y_test = dataset_scaler_service.scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1))

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    result = {
        "Group": name,
        "model": "RNN",
        "RMSE": rmse,
        "R2": r2,
        "Features": len(features)
    }
    results.append(result)

rnn_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(rnn_df)
rnn_df.to_csv("../results/rnn_experiment_5.csv", index=False)

/home/gabriel/Documents/ic_qc/rgpe/venv/lib64/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-07 21:09:43.969182: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


/home/gabriel/Documents/ic_qc/rgpe/venv/lib64/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


/home/gabriel/Documents/ic_qc/rgpe/venv/lib64/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/home/gabriel/Documents/ic_qc/rgpe/venv/lib64/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


/home/gabriel/Documents/ic_qc/rgpe/venv/lib64/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
                          Group model      RMSE        R2  Features
0                      J. Kampe   RNN  0.030335  0.985992        40
1            Random Forest (10)   RNN  0.034311  0.982079        10
2            mRMR (14 features)   RNN  0.035697  0.980603        14
3                 Gevrey Method   RNN  0.039187  0.976625        24
4             Random Forest (5)   RNN  0.039443  0.976317         5
5   Gevrey Method (20 features)   RNN  0.042129  0.972983        20
6            mRMR (10 features)   RNN  0.045022  0.969145        10
7            Random Forest Full   RNN  0.045149  0.968970        20
8   Gevrey Method (14 features)   RNN  0.046597  0.966948        14
9   Gevrey Method (12 features)   RNN  0.053143  0.957009        12
10  Gevrey Method (10 features)   RNN  0.079525  0.903730        10
11   Gevrey Method (8 features)   RNN  0.100306  0.846843         8
12                  Correlation   RNN  0.110983  0.812504        44
